# Matrixsteifigkeitsmethode – Tool

Dieses Notebook ist ein kompaktes FE-Tool für ebene Fachwerke.
Die gesamte Berechnungslogik ist in `fem_core.py` gekapselt; hier werden nur Modelldaten definiert und Ergebnisse ausgegeben.

In [ ]:
import numpy as np
from fem_core import assemble_K, solve_system
from fem_post import postprocessing, plot_results

## Input

Einheiten: Koordinaten und Längen in $[\text{mm}]$, Flächen in $[\text{mm}^2]$, E-Modul in $[\text{MPa}]$, Kräfte in $[\text{N}]$.

In [ ]:
nodal_coordinates = np.array([
    [   500.0,    900.0],   # Knoten 1
    [  1900.0,   1100.0],   # Knoten 2
    [  1900.0,    300.0],   # Knoten 3
])

elements = [
    [0, 1, "s1"],   # Stab 1
    [2, 1, "s1"],   # Stab 2
    [2, 0, "s1"],   # Stab 3
]

materials = {
    "s1": [210000.0],
    "s2": [210000.0],
    "s3": [210000.0],
    "s4": [210000.0],
    "s5": [210000.0],
}

sections = {
    "s1": [15.00, "s1"],
    "s2": [28.28, "s2"],
    "s3": [10.00, "s3"],
    "s4": [56.56, "s4"],
    "s5": [10.00, "s5"],
}

constraints = [
    [0, 0, 0.0],   # Knoten 1: x gesperrt
    [0, 1, 0.0],   # Knoten 1: y gesperrt
    [2, 1, 0.0],   # Knoten 3: y gesperrt
]

loads = [
    [1, 0, 1000.0],   # Knoten 2: F_x = 1000.0 N
]

## Core

In [ ]:
K            = assemble_K(nodal_coordinates, elements, sections, materials)
U, F, fixed  = solve_system(K, constraints, loads)
eps, sig, N  = postprocessing(U, nodal_coordinates, elements, sections, materials)

## Ergebnisse

In [ ]:
print("Verschiebungen und Kräfte:")
print(f"  {'DOF':>3}  {'U [mm]':>16}  {'F [N]':>12}")
print("  " + "─" * 36)
for k in range(len(U)):
    print(f"  {k+1:>3}  {U[k]:>+16.6e}  {F[k]:>+12.4f}")

print("\nStabkräfte:")
print(f"  {'Stab':>4}  {'ε [-]':>14}  {'σ [MPa]':>10}  {'N [N]':>10}")
print("  " + "─" * 46)
for e in range(len(elements)):
    print(f"  {e+1:>4}  {eps[e]:>14.6e}  {sig[e]:>10.4f}  {N[e]:>10.4f}")

## Visualisierung

In [ ]:
plot_results(nodal_coordinates, elements, constraints, loads, U, sig, scale=200)